In [ ]:
import pandas as pd
import numpy as np
import os
import pyarrow.parquet as pq

In [ ]:
def read_parquet_with_checks(file_path):
    print(f"File size: {os.path.getsize(file_path)} bytes")

    # Try to read just metadata
    try:
        metadata = pq.read_metadata(file_path)
        print("Metadata read successfully")
        print(f"Number of rows: {metadata.num_rows}")
        print(f"Number of columns: {metadata.num_columns}")
    except Exception as e:
        print(f"Error reading metadata: {e}")

    try:
      df= pd.read_parquet(
        file_path,
        engine='fastparquet'  # Instead of pyarrow
        )
    except Exception as e:
      print(f"Fastparquet error: {e}")    
    return df  

## 1.1 million textual reviews

In [ ]:
all_textual_reviews = read_parquet_with_checks("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/all_collected_text_reviews_25112025.parquet")
all_textual_reviews.info()

In [ ]:
#classified reviews
all_classified_reviews = pd.read_csv("//Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/cs_srt_labeled_26112025.csv")
all_classified_reviews.info()

In [ ]:
well_classified_reviews = read_parquet_with_checks("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/merged_700k_classified_with_ethnicity_13012026.parquet")
well_classified_reviews.info()

In [ ]:
#see what actually happened tot he 400k+ reviews that were not classified
not_classified = all_classified_reviews[~all_classified_reviews['review_id'].isin(well_classified_reviews['review_id'])]
not_classified.nunique()


In [ ]:
not_classified.groupby('social_encounter').agg({'racial_connotations':'value_counts',
'connotation_type':'value_counts', 'classification_status':'value_counts'})

## validating the 700k reviews

In [ ]:
validated_sample = pd.read_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/validating_classification/stratified_sample_validation_700k_validated.csv")
validated_sample.info()

In [ ]:
validated_sample.nunique()

In [ ]:
validated_sample.groupby('human_social_encounter').agg({'human_racial_connotation':'value_counts',
'human_connotation_type':'value_counts'}) 




In [ ]:
validated_sample.human_social_encounter.value_counts()

## time span for classified reviews

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np

COLLECT_START = pd.Timestamp("2025-08-01")
COLLECT_END   = pd.Timestamp("2025-10-31")


In [ ]:
def relative_to_days(text):
    if not isinstance(text, str):
        return None

    text = text.lower().strip()

    # remove "edited" prefix if present
    if text.startswith("edited "):
        text = text.replace("edited ", "", 1)

    # special cases
    if text in {"just now", "now", "today"}:
        return 0
    if text == "yesterday":
        return 1

    # handle "a"/"an"
    if text.startswith(("a ", "an ")):
        if "day" in text:
            return 1
        if "week" in text:
            return 7
        if "month" in text:
            return 30
        if "year" in text:
            return 365
        if "hour" in text or "minute" in text or "second" in text:
            return 0

    nums = re.findall(r"\d+", text)
    if not nums:
        return None

    n = int(nums[0])

    if "day" in text:
        return n
    if "week" in text:
        return n * 7
    if "month" in text:
        return n * 30
    if "year" in text:
        return n * 365
    if "hour" in text or "minute" in text or "second" in text:
        return 0

    return None


In [ ]:
well_classified_reviews["delta_days"] = (
    well_classified_reviews["review_time"]
    .apply(relative_to_days)
)

# Inspect failures before dropping
failed = well_classified_reviews[
    well_classified_reviews["delta_days"].isna()
]["review_time"].value_counts().head(10)

print(failed)


In [ ]:
well_classified_reviews["earliest_date"] = COLLECT_START - pd.to_timedelta(well_classified_reviews["delta_days"], unit="D")
well_classified_reviews["latest_date"]   = COLLECT_END   - pd.to_timedelta(well_classified_reviews["delta_days"], unit="D")

In [ ]:
bins = pd.date_range("2010-01-01", "2025-12-31", freq="YS")


In [ ]:
min_counts = []
max_counts = []

for i in range(len(bins) - 1):
    b_start = bins[i]
    b_end   = bins[i + 1]

    # minimum: reviews whose entire interval lies inside bin
    min_c = ((well_classified_reviews["earliest_date"] >= b_start) &
             (well_classified_reviews["latest_date"]   <= b_end)).sum()

    # maximum: reviews whose interval overlaps bin
    max_c = ((well_classified_reviews["latest_date"]   >= b_start) &
             (well_classified_reviews["earliest_date"] <= b_end)).sum()

    min_counts.append(min_c)
    max_counts.append(max_c)


In [ ]:
racialised_df = well_classified_reviews[
    well_classified_reviews["racial_connotations"] == 'yes'
]

subset_min = []
subset_max = []

for i in range(len(bins) - 1):
    b_start = bins[i]
    b_end   = bins[i + 1]

    min_c = ((racialised_df["earliest_date"] >= b_start) &
                                (racialised_df["latest_date"]   <= b_end)).sum()

    max_c = ((racialised_df["latest_date"]   >= b_start) &
             (racialised_df["earliest_date"] <= b_end)).sum()

    subset_min.append(min_c)
    subset_max.append(max_c)


In [ ]:
main_mid    = [(a + b) / 2 for a, b in zip(min_counts, max_counts)]
subset_mid = [(a + b) / 2 for a, b in zip(subset_min, subset_max)]


In [ ]:
import matplotlib.pyplot as plt

x = bins[:-1]

plt.figure(figsize=(12, 8))
plt.fill_between(x, min_counts, max_counts, alpha=0.3)
plt.plot(x, [(a + b) / 2 for a, b in zip(min_counts, max_counts)],
         linestyle="--",linewidth=2)

plt.ylabel("Review volume", fontsize=30)
plt.yticks(fontsize=30)
plt.xlabel("Year", fontsize=30)
plt.xticks(fontsize=30, rotation=45)
#plt.title("Plausible review volume over time (uncertainty-aware)")
plt.tight_layout()
plt.show()


In [ ]:
bins = pd.date_range("2010-01-01", "2025-12-31", freq="QS")

In [ ]:
min_counts = []
max_counts = []

for i in range(len(bins) - 1):
    b_start = bins[i]
    b_end   = bins[i + 1]

    min_c = ((well_classified_reviews["earliest_date"] >= b_start) &
             (well_classified_reviews["latest_date"]   <= b_end)).sum()

    max_c = ((well_classified_reviews["latest_date"]   >= b_start) &
             (well_classified_reviews["earliest_date"] <= b_end)).sum()

    min_counts.append(min_c)
    max_counts.append(max_c)


In [ ]:
racialised_df = well_classified_reviews[
    well_classified_reviews["racial_connotations"] == 'yes'
]

subset_min = []
subset_max = []

for i in range(len(bins) - 1):
    b_start = bins[i]
    b_end   = bins[i + 1]

    min_c = ((racialised_df["earliest_date"] >= b_start) &
             (racialised_df["latest_date"]   <= b_end)).sum()

    max_c = ((racialised_df["latest_date"]   >= b_start) &
             (racialised_df["earliest_date"] <= b_end)).sum()   
    subset_min.append(min_c)
    subset_max.append(max_c)


In [ ]:
plt.figure(figsize=(11, 4.8))

plt.fill_between(
    bins[:-1],
    min_counts,
    max_counts,
    alpha=0.25,
    label="All reviews (min–max)"
)

plt.plot(
    bins[:-1],
    main_mid,
    linestyle="--",
    linewidth=2.5,
    label="All reviews (mean)"
)

plt.plot(
    bins[:-1],
    subset_mid,
    linewidth=3,
    label="Subset of interest"
)

plt.xlabel("Time", fontsize=14)
plt.ylabel("Review volume", fontsize=14)

plt.xticks(
    bins[::4],  # label once per year
    [f"{d.year}" for d in bins[::4]],
    fontsize=12,
    rotation=0
)

plt.yticks(fontsize=12)

plt.legend(fontsize=12, frameon=False)
plt.tight_layout()
plt.show()


## predicting racial discourse

\item \textbf{RQ2 - Prediction and Explanation:} Can we predict spatial variation in racialized discourse from structural factors (pool attributes, demographics, political orientation)? What does model performance reveal about explanatory power and intervention sites?


First we need tocheck it there si a relationship between racial_discourse and other demographic, political and pool features. Are they statistically significant?
Can we have a multilevel predictive regression analysis?

In [ ]:
# data of interest 
racial_reviews = read_parquet_with_checks("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/final_merged_data_for_analysis_03122025.parquet")
racial_reviews.info()

In [ ]:
racial_reviews[['social_encounter','racial_connotations', 'connotation_type']].nunique()

In [ ]:
#demographics and political  data
demo_pol = pd.read_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/voters_demographic_datai.csv")
demo_pol.info()